# Testing Context Extraction with local LLM - Qwen2.5-coder:14b

In [2]:
# imports
from urllib import request
import json


In [3]:
# API
url = "http://localhost:11434/api/generate"

## Test Examples

In [ ]:
fileCode = """
START = 0
UPPER_BOUND = 100

def sum_n(n):
    return compute_sum(n)

def compute_sum(n):
    total = START
    for i in range(n):
        total += i
    return total


def sum_up_to_limit():
    return compute_sum(UPPER_BOUND)
"""

functionToAnalyze = "sum(n)"

fileCode1 = """
num_a = 10
num_b = 20
c = 30

def func1():
    return num_a + helper1()

def helper1():
    return num_b + helper2()

def helper2():
    return 42

def dummy1():
    test = 5
    return test
"""

functionToAnalyze1 = "func1()"

fileCode2 = """
limit = 10
offset = 3
factor = 2
unused = 999

def func2(n):
    if n > limit:
        return helperA(n) + offset
    else:
        return helperB(n)

def helperA(x):
    return x * factor

def helperB(x):
    return x + factor

def dummy2():
    return unused
"""

functionToAnalyze2 = "func2(n)"

fileCode3 = """
threshold = 100
step = 3
start = 1

def func3():
    value = start
    while value < threshold:
        value = helper3(value)
    return value

def helper3(x):
    return x + step

def dummy3():
    return threshold
"""

functionToAnalyze3 = "func3()"

fileCode4 = """
base = 0
decrement = 1

def func4(n):
    if is_base(n):
        return n
    return func4(n - decrement)

def is_base(x):
    return x <= base

def helper_unused():
    return decrement
"""

functionToAnalyze4 = "func4(n)"

fileCode5 = """
MAX = 100
MIN = 1
STEP = 5
factor = 2
unused = 999

def func5(n):
    total = 0
    for i in range(MIN, n, STEP):
        if i % 2 == 0:
            total += helper_even(i)
        else:
            total += helper_odd(i)
    while total < MAX:
        total = helper_increment(total)
    return total

def helper_even(x):
    return x * factor + helper_increment(x)

def helper_odd(x):
    if x % 3 == 0:
        return x // 2
    else:
        return x + factor

def helper_increment(value):
    if value >= MAX:
        return value
    return value + STEP

def dummy5():
    temp = 42
    return unused
"""

functionToAnalyze5 = "func5(n)"

fileCode6 = """
LIMIT = 50
OFFSET = 3
factor = 2
unused_global = 999

def func7():
    return "I am not used"

def func6(x):
    result = 0
    for i in range(x):
        if i % 2 == 0:
            result += even(i)
        elif i % 3 == 0:
            result += helper_multiple_three(i)
        else:
            result += helper_odd(i)
    return result

def even(n):
    return n * factor

def helper_odd(n):
    return n + OFFSET

def helper_multiple_three(n):
    if n > LIMIT:
        return LIMIT
    return n * 2

def dummy6():
    temp = 123
    return unused_global
"""

functionToAnalyze6 = "func6(x)"

fileCode7 = """
BASE = 10
STEP = 2
MAX = 100
factor = 3

def dummy7():
    return 42  # Nicht genutzt

def func7(n):
    total = 0
    while n > 0:
        total += process_step(n)
        n -= STEP
    return finalize(total)

def process_step(value):
    if value % 5 == 0:
        return helper5(value)
    elif value % 3 == 0:
        return helper3(value)
    else:
        return value + factor

def helper5(v):
    return v // 5

def helper3(v):
    return v // 3

def finalize(result):
    return result % MAX

def unused_helper():
    return 0
"""

functionToAnalyze7 = "func7(n)"

fileCode8 = """
LIMIT = 1000
FACTOR = 3
OFFSET = 7
BASE = 5
unused_global = 999

def dummy_unused1():
    return 0

def func8(n):
    result = BASE
    for i in range(n):
        if i % 2 == 0:
            result += helper_even(i)
        else:
            result += helper_odd(i)
        if result > LIMIT:
            result = cap_value(result)
    return finalize(result)

def helper_even(x):
    if x > OFFSET:
        return x * FACTOR
    else:
        return x + helper_nested(x)

def helper_odd(y):
    total = y
    while total < LIMIT:
        total += helper_even(total) // 2
        if total % 5 == 0:
            total -= helper_subtract(total)
    return total

def helper_nested(z):
    if z % 3 == 0:
        return z // 3 + helper_extra(z)
    return z + 1

def helper_extra(v):
    if v % 7 == 0:
        return v - 1
    return v + 2

def helper_subtract(val):
    return val - OFFSET

def cap_value(val):
    if val > LIMIT:
        return LIMIT
    return val

def finalize(final_result):
    return final_result % FACTOR

def dummy_unused2():
    temp = 123
    return unused_global
"""

functionToAnalyze8 = "func8(n)"



# Versuch 1

In [5]:
def create_context_v1(full_code, target_fn):
    prompt = f"""You are a static program analysis assistant. You reason over Python source code precisely and deterministically.

    Full source code:
    {full_code}

    Task:
    1. Identify ALL functions that are directly or indirectly called by {target_fn}.
    2. Identify ALL global variables used by those functions.
    3. Extract ONLY:
    - the definitions of those global variables
    - the definitions of those functions
    - the definition of the target function itself
    4. Sort the extracted functions topologically:
    - functions first
    - callers after callees

    Rules:
    - Output valid Python Code.
    - Do NOT include ```python blocks
    - Do NOT include unused functions or globals.
    - Do NOT include comments, explanations, or markdown.
    - Preserve exact Python syntax and indentation.
    """

    payload = {
        "model": "qwen2.5-coder:14b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted

In [6]:
result = create_context_v1(fileCode, functionToAnalyze)

In [7]:
from IPython.display import display, HTML

display(HTML(f"""
<div style="max-height:400px; overflow:auto; border:1px solid #ccc; padding:10px; white-space:pre;">
{result}
</div>
"""))

# Versuch 2

In [8]:
def find_called_functions_names(full_code, target_fn):
    prompt = f"""You are a deterministic static program analysis assistant.

    Task:
    Determine the set of functions that are directly or indirectly called by the target function.

    Definitions:
    - A function A is "called by" function B if B directly calls A, or B calls another function that calls A (transitively).

    Output requirements:
    - Return ONLY a JSON object with this exact schema: "called_functions": ["..."]
    - The list MUST include the target function itself.
    - Include ONLY functions that are directly or indirectly called by the target function.
    - Use function names exactly as defined in the source code.

    Inputs:
    Target function:
    {target_fn}

    Full source code:
    {full_code}
    """

    payload = {
        "model": "qwen2.5-coder:14b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted


def find_all_global_variables(full_code):
    prompt = f"""You are a deterministic static program analysis assistant.

    Task:
    Identify all global variables defined at module level in the given Python source code.

    Definitions:
    - A global variable is defined at module level (top-level of the file).
    - It is NOT defined inside any function, class, or lambda.

    Do NOT include:
    - Local variables inside functions or classes
    - Function names
    - Class names
    - Imported names

    Output requirements:
    - Return ONLY a JSON object with this exact schema: "all_variables": ["..."]
    - Use variable names exactly as defined in the source code.
    - Output ONLY JSON. No markdown, no code fences, no extra text.

    Inputs:
    Full source code:
    {full_code}
    """

    payload = {
        "model": "qwen2.5-coder:14b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted

# - Identify the definitions for the functions from 'Relevant Functions' in 'Full source code' and summarize them into one String.
def summarize_called_functions(full_code, called_functions):
    prompt = f"""You are a deterministic static program analysis assistant.

    Task:
    Extract ONLY the function definitions whose names are listed in the relevant function list.

    Definitions:
    - A relevant function definition is defined using 'def'.
    - The entire function body must be included exactly as written in the source code.
    - Extract ONLY those function definitions and nothing else.
    - Do NOT include ```python blocks

    Inputs:
    Relevant function names:
    {called_functions}

    Full source code:
    {full_code}

    Output requirements:
    - Output ONLY valid Python code consisting of the extracted function definitions.
    - Do NOT include any global variables.
    - Preserve the original indentation and line breaks exactly.
    - No markdown, no code fences, no explanations, no comments.
    - Do NOT include ```python blocks
    """

    payload = {
        "model": "qwen2.5-coder:14b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted

def topologically_sort_functions(called_functions_summary):
    prompt = f"""You are a deterministic static program analysis assistant.

    Task:
    Reorder the given function definitions into a valid topological order based on call dependencies within this set.

    Definitions:
    - A function A depends on function B if A calls B and B is also present in the given input set.
    - Consider only calls to functions included in the input set.
    - Do NOT include ```python blocks

    Inputs:
    Function definitions to reorder:
    {called_functions_summary}

    Output requirements:
    - Output ONLY Python code consisting of the same function definitions as the input.
    - Do NOT add, remove, rename, or edit any function.
    - Preserve the function bodies exactly, including indentation and line breaks.
    - No markdown, no code fences, no explanations, no comments.
    - Do NOT include ```python blocks
    """

    payload = {
        "model": "qwen2.5-coder:14b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    from urllib import request
    import json

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted


def find_all_variable_dependencies(called_functions_summary, global_variables):
    prompt = f"""You are a deterministic static program analysis assistant.

    Task:
    From a given list of global variables, return the subset that is referenced in the given source code.

    Definitions:
    - A global variable is considered "used" if its name appears as an identifier in the code (not inside strings/comments).
    - Only count names that exactly match entries in the provided list.
    - Do NOT infer or guess additional variables.

    Do NOT include:
    - Function names
    - Parameter names or local variables
    - Attributes such as obj.NAME (attribute access does not count as using global NAME)
    - Names that appear only inside strings or comments

    Inputs:
    Global variables (exact JSON array):
    {global_variables}

    Source code to analyze:
    {called_functions_summary}

    Output requirements:
    - Output ONLY a JSON array of variable names, e.g. ["var_a", "var_b"]
    - Include each variable at most once.
    - Use variable names exactly as given in the global variable list.
    - Output ONLY JSON. No markdown, no code fences, no extra text.
    """

    payload = {
        "model": "qwen2.5-coder:14b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted

def summarize_called_global_variables(full_code, called_global_variables):
    prompt = f"""You are a deterministic static program analysis assistant.

    Task:
    Extract ONLY the global variable definitions whose names are listed in the provided variable list.

    Definitions:
    - The variable name must be listed in the provided list.

    Do NOT include:
    - Any variables not listed in the provided list
    - Function definitions
    - Assignments inside functions or classes

    Inputs:
    Global variable names:
    {called_global_variables}

    Full source code:
    {full_code}
    
    Output requirements:
    - Output ONLY valid Python code consisting of the extracted variable definitions.
    - Preserve original indentation and line breaks exactly.
    - No markdown, no code fences, no explanations, no comments.
    - Do NOT include ```python blocks

    """


    payload = {
        "model": "qwen2.5-coder:14b",
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.0, "top_p": 1.0},
    }

    req = request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    formatted = raw.replace("\\n", "\n").strip()
    return formatted



In [9]:
fullFileCode = fileCode4
finalFunctionToAnalyse = functionToAnalyze4
called_functions = find_called_functions_names(finalFunctionToAnalyse, fullFileCode)
global_variables = find_all_global_variables(fullFileCode)
called_functions_summary = summarize_called_functions(fullFileCode, called_functions)
called_functions_summary_sorted = topologically_sort_functions(called_functions_summary)
called_global_variables = find_all_variable_dependencies(called_functions_summary, global_variables)
called_global_variables_summary = summarize_called_global_variables(fullFileCode, called_global_variables)
result = called_global_variables_summary + "\n" + "\n" + called_functions_summary_sorted

print("\n" + "="*30)
print("Called Functions")
print(called_functions)

print("\n" + "="*30)
print("All Global Variables")
print(global_variables)

print("\n" + "="*30)
print("Called Functions Summary")
print(called_functions_summary)



Called Functions
```json
{
  "called_functions": ["func4", "is_base"]
}
```

All Global Variables
{
  "all_variables": ["base", "decrement"]
}

Called Functions Summary
def func4(n):
    if is_base(n):
        return n
    return func4(n - decrement)

def is_base(x):
    return x <= base


In [10]:
print("\n" + "="*30)
print("Called Functions Summary sorted")
print(called_functions_summary_sorted)

print("\n" + "="*30)
print("Called Global Variables")
print(called_global_variables)

print("\n" + "="*30)
print("Called Global Variables Summary")
print(called_global_variables_summary)


Called Functions Summary sorted
```python
def is_base(x):
    return x <= base

def func4(n):
    if is_base(n):
        return n
    return func4(n - decrement)
```

Called Global Variables
["base", "decrement"]

Called Global Variables Summary
base = 0  
decrement = 1


In [11]:
from IPython.display import display, HTML

display(HTML(f"""
<div style="max-height:400px; overflow:auto; border:1px solid #ccc; padding:10px; white-space:pre;">
{result}
</div>
"""))